# Translating German to English using a pre-built ONNX from S3

This notebook shows the **fastest path** to running a neural machine
translation model inside **Teradata Vantage** with the Bring Your
Own Model (BYOM) framework: download a pre-built ONNX model and its
tokenizer directly from a public S3 bucket, deploy them with
`teradataml.save_byom`, and translate a set of German sentences to
English with a single SQL query.

This is a companion to the end-to-end notebook
[`opus_de_en_demo.ipynb`](opus_de_en_demo.ipynb), which walks
through the **full** pipeline including the conversion step from a
HuggingFace checkpoint to ONNX. If you don't need to re-convert —
because we've already published a pre-built artifact for the model
you want — start here. You skip:

- the HuggingFace hub download of the original checkpoint,
- the `torch.onnx.export` step on your machine (which needs PyTorch
  installed and a few minutes of CPU time),
- the local verification that the exported graph is BYOM-compatible.

What's left is a small download from a public S3 bucket and a
short series of `teradataml` calls — the BYOM-specific pieces of
the demo, with everything else stripped away.


## Install

This notebook uses `teradataml` (the official Teradata Python
client) to upload BLOBs and query the database. We also install
the `teradata-opus-translate` package — strictly speaking it isn't
required for the S3-download path (no `convert_model` is called
here), but it's the same companion package the conversion notebook
uses, so installing it now keeps both flows reachable from one
environment.

> **Note on outputs.** This notebook is committed with sanitised
> example outputs so you can read it top-to-bottom without a
> database connection. When you execute the cells against your own
> Teradata instance, the values you see (timings, server version,
> exact translations) will of course differ — the structure stays
> the same.


In [1]:
%pip install --quiet teradata-opus-translate teradataml

Note: you may need to restart the kernel to use updated packages.


## Why download from S3?

We publish ONNX-ready artifacts for the OPUS-MT translation
family to a **public S3 bucket** so you don't have to repeat the
conversion step on every machine that wants to deploy the model.
The bucket is `teradata-opus-translate-ce` in `us-east-1`, with
public-read access — no AWS credentials required, plain HTTPS GETs
work.

For each published model the bucket holds three files:

- `model-fp32.onnx` — the full-precision ONNX graph (~177 MiB
  for the OPUS-MT tiny family).
- `model-int8.onnx` — a dynamically-quantised int8 variant
  (~95 MiB) for setups where memory or download size matters
  more than the last few BLEU points.
- `tokenizer.json` — the HuggingFace `tokenizers`-format
  tokenizer file (~2 MiB).

The full catalog of published models is described in the
[S3 manifest](https://github.com/asmirnov-tba/teradata-opus-translate-ce/blob/main/data/s3_manifest.json),
which lists every available checkpoint with stable URLs and file
sizes. Browse it to discover models for other language pairs.

For this demo we use **`Helsinki-NLP/opus-mt_tiny_deu-eng`** —
the German-to-English tiny OPUS-MT checkpoint. Same architecture
as the full-size `opus-mt-de-en` model, smaller footprint, fast
on CPU.


## Setup

A small set of imports and the connection parameters for your
Teradata instance. Host, user, password, and target database are
read from environment variables so the notebook can be re-run
against any Teradata Vantage instance with `TD_MLDB.ONNXSeq2Seq`
available.


In [2]:
from __future__ import annotations

import os
import time
import urllib.request
from pathlib import Path

import pandas as pd

# ----------------------------------------------------------------
# Connection parameters. Set these for your Teradata instance,
# either by editing the placeholders below or by exporting the
# matching environment variables before launching the notebook.
# The placeholder strings are intentionally invalid hostnames so
# the connection step fails loudly if the values are not set.
# ----------------------------------------------------------------
TD_HOST = os.environ.get("TD_HOST", "<your-teradata-host>")
TD_USER = os.environ.get("TD_USER", "<your-user>")
TD_PASSWORD = os.environ.get("TD_PASSWORD", "<your-password>")

# Database that holds the BYOM tables. Pick any database your
# user has CREATE TABLE rights on.
TD_BYOM_DATABASE = os.environ.get("TD_BYOM_DATABASE", "OPUS_BYOM")

# Model identity. The short id is used as the row id inside
# Teradata's BYOM tables. We add a `-from-s3` suffix so this
# notebook's BYOM rows do not collide with rows that the
# conversion-from-HF notebook may have left behind.
S3_BUCKET = "teradata-opus-translate-ce"
S3_REGION = "us-east-1"
S3_OPSET = 14
HF_MODEL_ID = "Helsinki-NLP/opus-mt_tiny_deu-eng"
BYOM_MODEL_ID = "opus-mt_tiny_deu-eng-from-s3"

S3_BASE = f"https://{S3_BUCKET}.s3.{S3_REGION}.amazonaws.com/opus-translate/{S3_OPSET}/{HF_MODEL_ID}"
ONNX_URL = f"{S3_BASE}/model-fp32.onnx"
TOKENIZER_URL = f"{S3_BASE}/tokenizer.json"

# Local cache. The fp32 ONNX is ~177 MiB so we keep it on disk
# between runs.
CACHE_DIR = Path.home() / ".cache" / "teradata-opus-translate" / "s3"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = CACHE_DIR / "opus-mt_tiny_deu-eng.onnx"
TOKENIZER_PATH = CACHE_DIR / "opus-mt_tiny_deu-eng.tokenizer.json"

print(f"Target Teradata host: {TD_HOST}")
print(f"BYOM database:        {TD_BYOM_DATABASE}")
print(f"HF model id:          {HF_MODEL_ID}")
print(f"BYOM row id:          {BYOM_MODEL_ID}")
print(f"S3 base URL:          {S3_BASE}")


Target Teradata host: <your-teradata-host>
BYOM database:        OPUS_BYOM
HF model id:          Helsinki-NLP/opus-mt_tiny_deu-eng
BYOM row id:          opus-mt_tiny_deu-eng-from-s3
S3 base URL:          https://teradata-opus-translate-ce.s3.us-east-1.amazonaws.com/opus-translate/14/Helsinki-NLP/opus-mt_tiny_deu-eng


## Step 1 — Download the model and tokenizer from S3

The bucket is public-read so a plain HTTPS GET is enough — no AWS
credentials, no `boto3`, no `requests` dependency. Python's
standard-library `urllib.request.urlretrieve` does the job.

We download both files into a local cache directory. On a re-run
the cached files are reused unchanged, so the network step only
happens once per machine.


In [3]:
def _download_if_missing(url: str, dest: Path) -> int:
    """Fetch *url* to *dest* if not already cached. Return file size in bytes."""
    if dest.exists():
        size = dest.stat().st_size
        print(f"  cached  {dest.name:42s} ({size:>12,} bytes)")
        return size
    print(f"  fetching {dest.name:41s} from {url}")
    t0 = time.perf_counter()
    urllib.request.urlretrieve(url, dest)
    elapsed = time.perf_counter() - t0
    size = dest.stat().st_size
    print(f"  done    {dest.name:42s} ({size:>12,} bytes, {elapsed:.1f}s)")
    return size


print("Downloading artifacts from S3 ...")
onnx_size = _download_if_missing(ONNX_URL, ONNX_PATH)
tokenizer_size = _download_if_missing(TOKENIZER_URL, TOKENIZER_PATH)
print()
print(f"ONNX model    : {onnx_size:,} bytes ({onnx_size / (1024 * 1024):.1f} MiB)")
print(f"Tokenizer     : {tokenizer_size:,} bytes ({tokenizer_size / 1024:.1f} KiB)")


  fetching opus-mt_tiny_deu-eng.onnx                 from https://teradata-opus-translate-ce.s3.us-east-1.amazonaws.com/opus-translate/14/Helsinki-NLP/opus-mt_tiny_deu-eng/model-fp32.onnx
  done    opus-mt_tiny_deu-eng.onnx                  ( 177,301,730 bytes, 6.4s)
  fetching opus-mt_tiny_deu-eng.tokenizer.json       from https://teradata-opus-translate-ce.s3.us-east-1.amazonaws.com/opus-translate/14/Helsinki-NLP/opus-mt_tiny_deu-eng/tokenizer.json
  done    opus-mt_tiny_deu-eng.tokenizer.json        (   2,104,618 bytes, 0.3s)

ONNX model    : 177,301,730 bytes (169.1 MiB)
Tokenizer     : 2,104,618 bytes (2055.3 KiB)


## Step 2 — Open the Teradata connection

`teradataml` keeps a single thread-local context per session.
`create_context` returns it; `remove_context` (in the cleanup
section) closes it cleanly so the runtime drops any temporary
`ml__*` views and tables it built up during the session.


In [4]:
from teradataml import (
    DataFrame as TdDataFrame,
    copy_to_sql,
    create_context,
    execute_sql,
    remove_context,
    save_byom,
)

ctx = create_context(host=TD_HOST, username=TD_USER, password=TD_PASSWORD)

td_version = execute_sql(
    "SELECT InfoData FROM DBC.DBCInfoV WHERE InfoKey = 'VERSION'"
).fetchone()
print(f"Connected to Teradata {td_version[0]} as {TD_USER}@{TD_HOST}")


Connected to Teradata 20.00.29.72 as <your-user>@<your-teradata-host>


## Step 3 — Deploy the model and tokenizer to BYOM

BYOM stores model artifacts as `BLOB` rows in simple two-column
tables. We use the canonical schema `(model_id VARCHAR, model BLOB)`
that `teradataml`'s `save_byom()` helper expects, with one table
per artifact kind:

- `OPUS_BYOM.onnx_models` — one row per ONNX model.
- `OPUS_BYOM.opus_tokenizers` — one row per tokenizer.

`save_byom()` does not overwrite an existing row, so we `DELETE`
any previous row of the same id first. This makes the cell
idempotent — re-running the notebook against the same database
just refreshes the deployed bytes.

A small naming detail: BYOM's `ONNXSeq2Seq` operator expects the
input column on the `TokenizerTable` clause to be named
`tokenizer`. Because we store the BLOB under the canonical
`save_byom` column name `model`, we alias it on `SELECT` in the
SQL invocation a few cells down (`SELECT model AS tokenizer ...`).


In [5]:
TOKENIZER_TABLE = "opus_tokenizers"

# --- model -----------------------------------------------------------
try:
    execute_sql(
        f"DELETE FROM {TD_BYOM_DATABASE}.onnx_models "
        f"WHERE model_id = '{BYOM_MODEL_ID}'"
    )
except Exception as exc:
    if "[Error 3807]" not in str(exc):  # 3807 = "object does not exist"
        raise

print(f"Saving ONNX model to {TD_BYOM_DATABASE}.onnx_models ...")
save_byom(
    model_id=BYOM_MODEL_ID,
    model_file=str(ONNX_PATH),
    table_name="onnx_models",
    schema_name=TD_BYOM_DATABASE,
)
print("  loaded.")

# --- tokenizer -------------------------------------------------------
try:
    execute_sql(
        f"DELETE FROM {TD_BYOM_DATABASE}.{TOKENIZER_TABLE} "
        f"WHERE model_id = '{BYOM_MODEL_ID}'"
    )
except Exception as exc:
    if "[Error 3807]" not in str(exc):
        raise

print(f"Saving tokenizer to {TD_BYOM_DATABASE}.{TOKENIZER_TABLE} ...")
save_byom(
    model_id=BYOM_MODEL_ID,
    model_file=str(TOKENIZER_PATH),
    table_name=TOKENIZER_TABLE,
    schema_name=TD_BYOM_DATABASE,
)
print("  loaded.")


Saving ONNX model to OPUS_BYOM.onnx_models ...
Model is saved.
  loaded.
Saving tokenizer to OPUS_BYOM.opus_tokenizers ...
Model is saved.
  loaded.


## Step 4 — Load the demo inputs

A small set of German sentences chosen to exercise different
sides of the translation model: a casual greeting, a business
sentence with named numbers, a technical sentence, a German
idiom, a sentence with named entities and a date, a sentence
with nested subordinate clauses, and three more for variety.

`copy_to_sql` collapses CREATE TABLE + INSERT into a single
`teradataml` call; `if_exists='replace'` makes the cell
idempotent.


In [6]:
DEMO_SENTENCES: list[tuple[str, str]] = [
    ("greeting",    "Hallo Welt, wie geht es dir heute?"),
    ("business",    "Die Quartalszahlen werden nächste Woche um neun Uhr veröffentlicht."),
    ("technical",   "Der neue Compiler optimiert den Maschinencode für moderne Prozessoren."),
    ("idiom",       "Ich verstehe nur Bahnhof."),
    ("entity",      "Albert Einstein wurde 1879 in Ulm geboren und entwickelte die Relativitätstheorie."),
    ("long",        "Obwohl es stark regnete, beschlossen wir, den Spaziergang im Park "
                    "fortzusetzen, weil das Wetter laut Vorhersage am Nachmittag besser werden sollte."),
    ("question",    "Können Sie mir bitte den Weg zum Bahnhof zeigen?"),
    ("food",        "Zum Frühstück gab es frisches Brot, Käse und einen starken Kaffee."),
    ("weather",     "Heute Morgen lag dichter Nebel über den Feldern."),
]

inputs_df = pd.DataFrame(DEMO_SENTENCES, columns=["id", "source_de"])

DEMO_INPUT_TABLE = "s3_demo_inputs"
DEMO_INPUT_QUALIFIED = f"{TD_BYOM_DATABASE}.{DEMO_INPUT_TABLE}"

copy_to_sql(
    df=inputs_df.rename(columns={"source_de": "txt"}),
    table_name=DEMO_INPUT_TABLE,
    schema_name=TD_BYOM_DATABASE,
    primary_index="id",
    if_exists="replace",
)
print(f"Loaded {len(inputs_df)} rows into {DEMO_INPUT_QUALIFIED}")
inputs_df


Loaded 9 rows into OPUS_BYOM.s3_demo_inputs


         id                                          source_de
0  greeting                 Hallo Welt, wie geht es dir heute?
1  business  Die Quartalszahlen werden nächste Woche um neu...
2 technical  Der neue Compiler optimiert den Maschinencode ...
3     idiom                          Ich verstehe nur Bahnhof.
4    entity  Albert Einstein wurde 1879 in Ulm geboren und ...
5      long  Obwohl es stark regnete, beschlossen wir, den ...
6  question        Können Sie mir bitte den Weg zum Bahnhof zeigen?
7      food  Zum Frühstück gab es frisches Brot, Käse und e...
8   weather   Heute Morgen lag dichter Nebel über den Feldern.


## Step 5 — Translate with `TD_MLDB.ONNXSeq2Seq`

This is the centerpiece. A single SQL `SELECT` invokes the
`TD_MLDB.ONNXSeq2Seq` table operator, which loads the deployed
model and tokenizer, encodes each input row, runs beam-search
generation inside the database, and returns the decoded English
text. The same query would translate ten million rows by changing
only the input table.

The operator takes three input streams and a set of named
parameters. We build the SQL as a plain Python string so you can
read every clause before it runs:

- **`Accumulate('id')`** — columns from the input table that pass
  straight through to the output.
- **`ModelOutputTensor('sequences')`** — which output tensor of the
  ONNX graph is returned. `sequences` is the decoded token-id
  tensor; BYOM converts it to text via the tokenizer.
- **`SkipSpecialTokens('true')`** — drop `<pad>`, `</s>` and
  similar special tokens from the decoded output.
- **`OutputLength(1024)`** — maximum bytes per translated string
  in the result column.
- **`EnableMemoryCheck('false')`** — bypass the runtime guard that
  reserves memory ahead of model load.
- **`OverwriteCachedModel('*')`** — forces a fresh load of the
  model BLOB on this query.
- **`Const_*` cluster** — generation hyperparameters mapped one-to-one
  onto the standard HuggingFace `model.generate(...)` arguments:
  - `Const_min_length(1)` / `Const_max_length(64)` — bounds on the
    number of generated tokens per row.
  - `Const_num_beams(4)` — beam-search width.
  - `Const_length_penalty(1.0)` / `Const_repetition_penalty(1.0)` —
    neutral generation defaults.

Note that `num_return_sequences` is **not** part of the
`Const_*` cluster — it is baked into the produced ONNX graph as
a constant, so each row always returns exactly one translation.


In [7]:
ONNXSEQ2SEQ_SQL = f"""\
SELECT id, sequences
FROM TD_MLDB.ONNXSeq2Seq(
    ON (SELECT id, txt FROM {DEMO_INPUT_QUALIFIED}) AS InputTable
    ON (SELECT model_id, model
        FROM {TD_BYOM_DATABASE}.onnx_models
        WHERE model_id = '{BYOM_MODEL_ID}') AS ModelTable DIMENSION
    ON (SELECT model AS tokenizer
        FROM {TD_BYOM_DATABASE}.{TOKENIZER_TABLE}
        WHERE model_id = '{BYOM_MODEL_ID}') AS TokenizerTable DIMENSION
    USING
        Accumulate('id')
        ModelOutputTensor('sequences')
        SkipSpecialTokens('true')
        OutputLength(1024)
        EnableMemoryCheck('false')
        OverwriteCachedModel('*')
        Const_min_length(1)
        Const_max_length(64)
        Const_num_beams(4)
        Const_length_penalty(1.000000)
        Const_repetition_penalty(1.000000)
) AS t
"""

print(ONNXSEQ2SEQ_SQL)


SELECT id, sequences
FROM TD_MLDB.ONNXSeq2Seq(
    ON (SELECT id, txt FROM OPUS_BYOM.s3_demo_inputs) AS InputTable
    ON (SELECT model_id, model
        FROM OPUS_BYOM.onnx_models
        WHERE model_id = 'opus-mt_tiny_deu-eng-from-s3') AS ModelTable DIMENSION
    ON (SELECT model AS tokenizer
        FROM OPUS_BYOM.opus_tokenizers
        WHERE model_id = 'opus-mt_tiny_deu-eng-from-s3') AS TokenizerTable DIMENSION
    USING
        Accumulate('id')
        ModelOutputTensor('sequences')
        SkipSpecialTokens('true')
        OutputLength(1024)
        EnableMemoryCheck('false')
        OverwriteCachedModel('*')
        Const_min_length(1)
        Const_max_length(64)
        Const_num_beams(4)
        Const_length_penalty(1.000000)
        Const_repetition_penalty(1.000000)
) AS t



Run the query and pull the results into a pandas DataFrame.
`DataFrame.from_query` lets `teradataml` materialise the result
lazily and render it inline, so we don't need a manual cursor.


In [8]:
print("Executing ONNXSeq2Seq ...")
t_td_start = time.perf_counter()
results = TdDataFrame.from_query(ONNXSEQ2SEQ_SQL)
results_pdf = results.to_pandas(all_rows=True)
td_elapsed = time.perf_counter() - t_td_start
print(f"Returned {len(results_pdf)} rows in {td_elapsed:.2f}s "
      f"({td_elapsed / max(len(results_pdf), 1):.2f}s per row)")


Executing ONNXSeq2Seq ...
Returned 9 rows in 4.81s (0.53s per row)


## Step 6 — The results

The translations as they came back from Teradata, laid out next
to the original German.


In [9]:
def _to_str(v: object) -> str:
    return v.decode("utf-8") if isinstance(v, (bytes, bytearray)) else str(v)

td_text: dict[str, str] = {
    _to_str(row["id"]): _to_str(row["sequences"])
    for _, row in results_pdf.iterrows()
}

display_df = pd.DataFrame(
    [(rid, src, td_text.get(rid, "<MISSING>")) for rid, src in DEMO_SENTENCES],
    columns=["id", "source_de", "teradata_en"],
)
pd.set_option("display.max_colwidth", 200)
display_df


          id                                                                                                                                             source_de  \
0   greeting                                                                                                                   Hallo Welt, wie geht es dir heute?   
1   business                                                                                  Die Quartalszahlen werden nächste Woche um neun Uhr veröffentlicht.   
2  technical                                                                               Der neue Compiler optimiert den Maschinencode für moderne Prozessoren.   
3      idiom                                                                                                                            Ich verstehe nur Bahnhof.   
4     entity                                                                   Albert Einstein wurde 1879 in Ulm geboren und entwickelte die Relativitätstheorie.   
5       l

Note the **idiom** row. `Ich verstehe nur Bahnhof.` is a German
expression that literally means "I only understand train station"
and idiomatically means "It's all Greek to me." The model produces
a faithful literal translation — a useful reminder that statistical
translation captures syntax and vocabulary cleanly but does not
always rewrite idioms into target-language equivalents. This is
behaviour of the underlying OPUS-MT checkpoint, not of BYOM.


## Cleanup

Drop the demo input table and remove the BYOM model and tokenizer
rows so the database is left in the same state it was in before
this notebook ran. The locally cached files under
`~/.cache/teradata-opus-translate/s3/` are left in place — they
take ~170 MiB and let you re-run this notebook without
re-downloading. Delete them manually if you want a fully clean
slate.


In [10]:
# --- delete the BYOM model and tokenizer rows ----------------------
execute_sql(
    f"DELETE FROM {TD_BYOM_DATABASE}.onnx_models "
    f"WHERE model_id = '{BYOM_MODEL_ID}'"
)
execute_sql(
    f"DELETE FROM {TD_BYOM_DATABASE}.{TOKENIZER_TABLE} "
    f"WHERE model_id = '{BYOM_MODEL_ID}'"
)
print(
    f"Removed BYOM rows for {BYOM_MODEL_ID!r} from "
    f"{TD_BYOM_DATABASE}.onnx_models and "
    f"{TD_BYOM_DATABASE}.{TOKENIZER_TABLE}"
)

# --- drop the demo input table -------------------------------------
execute_sql(f"DROP TABLE {DEMO_INPUT_QUALIFIED}")
print(f"Dropped {DEMO_INPUT_QUALIFIED}")

# --- close the Teradata session ------------------------------------
remove_context()
print("Teradata session closed.")


Removed BYOM rows for 'opus-mt_tiny_deu-eng-from-s3' from OPUS_BYOM.onnx_models and OPUS_BYOM.opus_tokenizers
Dropped OPUS_BYOM.s3_demo_inputs
Teradata session closed.


## What this demonstrated

- A pre-built ONNX model and matching tokenizer downloaded from
  the public `teradata-opus-translate-ce` S3 bucket — no AWS
  credentials, no `boto3`, no local conversion step.
- Both artifacts deployed as BLOBs inside a Teradata database with
  `teradataml.save_byom`.
- Nine German sentences translated to English with a **single SQL
  query** running entirely inside Teradata — no Python sidecar,
  no data movement.

For the full picture — how the ONNX file was produced from the
original HuggingFace checkpoint, including the `torch.onnx.export`
step and the parity check against `MarianMTModel.generate()` — see
the companion notebook
[`opus_de_en_demo.ipynb`](opus_de_en_demo.ipynb).

For other available pre-built models, browse the
[S3 manifest](https://github.com/asmirnov-tba/teradata-opus-translate-ce/blob/main/data/s3_manifest.json).
The pattern in this notebook works unchanged for any of them — change the
`HF_MODEL_ID` constant and re-run.
